# Предсказание цены квартиры

В этой тетрадке модель обучается уже на объявлениях, которые собрал парсер с Яндекс Недвижимости. Данных пока немного, поэтому качество не идеальное, но это уже ближе к настоящей задаче.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Загрузка данных

In [ ]:
df = pd.read_csv('../data/raw/flats_from_sites.csv', quotechar=chr(39))
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Чистка данных

In [ ]:
needed_columns = [
    'city',
    'price',
    'area',
    'rooms',
    'floor',
    'total_floors',
    'distance_from_center',
    'price_per_meter',
    'is_first_floor',
    'is_last_floor',
    'floor_ratio',
]

df = df[needed_columns].copy()
df = df.dropna()

df = df[df['price'] > 1000000]
df = df[df['area'] > 10]
df = df[df['rooms'] > 0]
df = df[df['floor'] > 0]
df = df[df['total_floors'] >= df['floor']]

df['price_per_meter'] = df['price_per_meter'].fillna((df['price'] / df['area']).round(2))
df['is_first_floor'] = df['is_first_floor'].fillna((df['floor'] == 1).astype(int))
df['is_last_floor'] = df['is_last_floor'].fillna((df['floor'] == df['total_floors']).astype(int))
df['floor_ratio'] = df['floor_ratio'].fillna((df['floor'] / df['total_floors']).round(3))
df = df[df['price_per_meter'].between(150000, 2500000)]

df.head()

In [ ]:
df.describe()

## Небольшой анализ

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['price'], bins=30)
plt.title('Распределение цены')
plt.xlabel('Цена')
plt.ylabel('Количество')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x='area', y='price')
plt.title('Цена и площадь')
plt.xlabel('Площадь')
plt.ylabel('Цена')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='Blues')
plt.title('Корреляции числовых признаков')
plt.show()

## Обучение модели

In [ ]:
target = 'price'

features = [
    'city',
    'area',
    'rooms',
    'floor',
    'total_floors',
    'distance_from_center',
    'is_first_floor',
    'is_last_floor',
    'floor_ratio',
]

cat_features = [0]

X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
params = {
    'iterations': 400,
    'learning_rate': 0.05,
    'depth': 5,
    'random_seed': 42,
    'verbose': 100,
    'train_dir': '../catboost_info',
}

model = CatBoostRegressor(**params, loss_function='RMSE')
low_model = CatBoostRegressor(**params, loss_function='Quantile:alpha=0.1')
high_model = CatBoostRegressor(**params, loss_function='Quantile:alpha=0.9')

In [ ]:
model.fit(X_train, y_train, cat_features=cat_features)
low_model.fit(X_train, y_train, cat_features=cat_features)
high_model.fit(X_train, y_train, cat_features=cat_features)

## Метрики

In [ ]:
pred = model.predict(X_test)
low_pred = low_model.predict(X_test)
high_pred = high_model.predict(X_test)

interval_low = np.minimum(low_pred, high_pred)
interval_high = np.maximum(low_pred, high_pred)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100
inside_interval = ((y_test >= interval_low) & (y_test <= interval_high)).mean() * 100

print('MAE:', round(mae, 2))
print('RMSE:', round(rmse, 2))
print('R2:', round(r2, 3))
print('MAPE:', round(mape, 2), '%')
print('Попадание в интервал:', round(inside_interval, 2), '%')

In [ ]:
results = pd.DataFrame({
    'real_price': y_test.values,
    'prediction': pred,
    'low': interval_low,
    'high': interval_high,
})

results.head(10)

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=results['real_price'], y=results['prediction'])
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.title('Реальная и предсказанная цена')
plt.show()

## Сохранение моделей

In [ ]:
model.save_model('../models/catboost_model.cbm')
low_model.save_model('../models/catboost_low.cbm')
high_model.save_model('../models/catboost_high.cbm')

## Пример прогноза

In [ ]:
def predict_price_from_params(flat):
    data = pd.DataFrame([flat])[features]

    price = model.predict(data)[0]
    low_price = low_model.predict(data)[0]
    high_price = high_model.predict(data)[0]

    return round(price), round(min(low_price, high_price)), round(max(low_price, high_price))


example_flat = {
    'city': 'Москва',
    'area': 60,
    'rooms': 2,
    'floor': 7,
    'total_floors': 16,
    'distance_from_center': 8,
    'is_first_floor': 0,
    'is_last_floor': 0,
    'floor_ratio': round(7 / 16, 3),
}

price, low_price, high_price = predict_price_from_params(example_flat)

print('Предсказанная цена:', price)
print('Интервал:', low_price, '-', high_price)